In [4]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Number of records
n = 40000

# Create datetime range (hourly data)
timestamps = pd.date_range(start="2022-01-01", periods=n, freq="h")


# Create base load pattern (daily + seasonal)
hours = timestamps.hour
days = timestamps.dayofyear

# Simulate load (MW)
load = (
    30000
    + 5000 * np.sin(2 * np.pi * hours / 24)  # daily pattern
    + 3000 * np.sin(2 * np.pi * days / 365)  # seasonal pattern
    + np.random.normal(0, 1000, n)          # noise
)

# Weather data
temperature = (
    10
    + 10 * np.sin(2 * np.pi * days / 365)
    + np.random.normal(0, 2, n)
)

humidity = np.clip(70 + np.random.normal(0, 10, n), 30, 100)
wind_speed = np.abs(np.random.normal(5, 2, n))

# Time features
hour = timestamps.hour
day_of_week = timestamps.dayofweek
is_weekend = (day_of_week >= 5).astype(int)

# Create DataFrame
df = pd.DataFrame({
    "timestamp": timestamps,
    "load_MW": load,
    "temperature_C": temperature,
    "humidity": humidity,
    "wind_speed": wind_speed,
    "hour": hour,
    "day_of_week": day_of_week,
    "is_weekend": is_weekend
})

# Create lag features
df["lag_1"] = df["load_MW"].shift(1)
df["lag_24"] = df["load_MW"].shift(24)

# Rolling mean
df["rolling_mean_3"] = df["load_MW"].rolling(window=3).mean()

# Introduce some missing values (real-world messiness)
for col in ["load_MW", "temperature_C"]:
    df.loc[np.random.choice(n, size=200, replace=False), col] = np.nan

# Save to CSV
df.to_csv("smart_grid_load_dataset.csv", index=False)

print("Dataset created successfully!")
df.head(50)

Dataset created successfully!


,timestamp,load_MW,temperature_C,humidity,wind_speed,hour,day_of_week,is_weekend,lag_1,lag_24,rolling_mean_3
0,2022-01-01 00:00:00,30548.354221,9.228418,76.640011,4.562699,0,5,1,NaN,NaN,NaN
1,2022-01-01 01:00:00,31207.470993,12.197539,67.038840,5.765877,1,5,1,30548.354221,NaN,NaN
2,2022-01-01 02:00:00,33199.328607,9.775760,69.334136,6.470284,2,5,1,31207.470993,NaN,31651.717940
3,2022-01-01 03:00:00,35110.203831,10.353272,68.246176,2.633271,3,5,1,33199.328607,NaN,33172.334477
4,2022-01-01 04:00:00,34147.613713,11.606915,75.140537,5.201478,4,5,1,35110.203831,NaN,34152.382050
5,2022-01-01 05:00:00,34647.132243,10.054207,72.861305,4.198075,5,5,1,34147.613713,NaN,34634.983262
6,2022-01-01 06:00:00,36630.852884,6.536438,89.063199,3.327399,6,5,1,34647.132243,NaN,35141.866280
7,2022-01-01 07:00:00,35648.703929,12.253309,66.158565,3.716587,7,5,1,36630.852884,NaN,35642.229685
8,2022-01-01 08:00:00,33912.292701,12.681991,87.928523,5.996411,8,5,1,35648.703929,NaN,35397.283171
9,2022-01-01 09:00:00,34129.734018,6.509901,51.882562,2.056190,9,5,1,33912.292701,NaN,34563.576883
